# Image Classification using CNN

This notebook classifies **cats and dogs** using a Convolutional Neural Network (CNN).

**Pipeline:**
1. Download Dataset
2. Data Preparation and Preprocessing
3. Build the CNN Model
4. Model Training
5. Visualize Results
6. Predictions on Test Images

## 1. Environment Setup

In [ ]:
# Verify GPU Availability in Google Colab
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

# If no GPU is detected:
# Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4)

In [ ]:
# Clone the Project Repository
import os

REPO_URL = 'https://github.com/ozodbek-bosimov/image-classification-using-cnn.git'
PROJECT_DIR = '/content/image-classification-using-cnn'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
    print('Project cloned')
else:
    !cd {PROJECT_DIR} && git pull
    print('Project updated')

## 2. Download the Dataset

In this step, we download the necessary dataset directly from Google Drive.

In [ ]:
# Download dataset
import os
import zipfile

FILE_ID = '19inwa0n1W4DZamjCOm5XAlztvqG_xkjP'
LOCAL_ZIP_PATH = f'{PROJECT_DIR}/dataset.zip'

os.makedirs(PROJECT_DIR, exist_ok=True)

if not os.path.exists(f'{PROJECT_DIR}/train'):
    print('Downloading dataset from Google Drive...')
    !gdown {FILE_ID} -O {LOCAL_ZIP_PATH} --quiet

    print('Extracting the main ZIP archive...')
    with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(PROJECT_DIR)
    os.remove(LOCAL_ZIP_PATH)

    print('Extracting the nested ZIP archives (train/test)...')
    for zip_name in ['train.zip', 'test1.zip']:
        inner_zip = os.path.join(PROJECT_DIR, zip_name)
        if os.path.exists(inner_zip):
            with zipfile.ZipFile(inner_zip, 'r') as zip_ref:
                zip_ref.extractall(PROJECT_DIR)
            os.remove(inner_zip)

    print('Dataset is fully prepared!')
else:
    print('Dataset is already prepared!')

# Verify dataset structure
try:
    train_count = len(os.listdir(f'{PROJECT_DIR}/train'))
    test_count = len(os.listdir(f'{PROJECT_DIR}/test1'))
    print(f'Train images count: {train_count}')
    print(f'Test images count: {test_count}')
except FileNotFoundError:
    print('An error occurred: Required folders were not found.')

## 3. Import Project Modules

In [ ]:
import sys
sys.path.insert(0, f'{PROJECT_DIR}/Code')

import constants as CONST
from data_prep import prep_and_load_data, get_size_statistics
from model import get_model

import numpy as np
import cv2
import copy
import pickle
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping

# Update directory paths for the Colab environment
CONST.TRAIN_DIR = f'{PROJECT_DIR}/train'
CONST.TEST_DIR = f'{PROJECT_DIR}/test1'

print(f'Train dir: {CONST.TRAIN_DIR}')
print(f'Test dir:  {CONST.TEST_DIR}')
print(f'IMG_SIZE:  {CONST.IMG_SIZE}')
print(f'DATA_SIZE: {CONST.DATA_SIZE}')

## 4. Data Exploration

In [ ]:
# Generate image size statistics
get_size_statistics()

In [ ]:
# Display sample images from the training set
import random

sample_images = random.sample(os.listdir(CONST.TRAIN_DIR), 10)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for idx, img_name in enumerate(sample_images):
    img_path = os.path.join(CONST.TRAIN_DIR, img_name)
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    label = 'Cat' if img_name.startswith('cat') else 'Dog'
    
    ax = axes[idx // 5, idx % 5]
    ax.imshow(img)
    ax.set_title(f'{label} ({img.shape[0]}x{img.shape[1]})')
    ax.axis('off')

plt.suptitle('Sample Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Data Preparation

In [ ]:
%%time
# Load and preprocess images
data = prep_and_load_data()

In [ ]:
# Create Train / Validation split
train_size = int(len(data) * CONST.SPLIT_RATIO)

train_data = data[:train_size]
test_data = data[train_size:]

train_images = np.array([i[0] for i in train_data], dtype=np.float32)
train_labels = np.array([i[1] for i in train_data], dtype=np.float32)

test_images = np.array([i[0] for i in test_data], dtype=np.float32)
test_labels = np.array([i[1] for i in test_data], dtype=np.float32)

print(f'Train images: {train_images.shape}')
print(f'Train labels: {train_labels.shape}')
print(f'Test images:  {test_images.shape}')
print(f'Test labels:  {test_labels.shape}')

# Clear variables to free up RAM
del data, train_data, test_data

# Check the class distribution
cat_count = int(train_labels[:, 0].sum())
dog_count = int(train_labels[:, 1].sum())
print(f'\nCats: {cat_count}, Dogs: {dog_count}')

## 6. Build the CNN Model

In [ ]:
model = get_model()
model.summary()

## 7. Model Training

In [ ]:
from google.colab import drive
import os
import pickle
from tensorflow.keras.models import load_model
import time

# Mount Google Drive for saving/loading the model
drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/ML_Models/cats_vs_dogs'
model_path = f'{save_dir}/cats_vs_dogs_cnn.h5'
history_path = f'{save_dir}/training_history.pkl'

if os.path.exists(model_path):
    print(f'Loading pre-trained model from Google Drive... ({model_path})')
    model = load_model(model_path)
    print('Model loaded! Training will be skipped.')
    
    class DummyHistory:
        history = None
    history = DummyHistory()
    if os.path.exists(history_path):
        with open(history_path, 'rb') as f:
            history.history = pickle.load(f)
            print('Previous training history was also loaded from Google Drive.')
else:
    print('Model not found in Google Drive. Starting training from scratch...')
    
    log_dir = f'{PROJECT_DIR}/logs/{int(time.time())}'
    tensorboard_cb = TensorBoard(log_dir=log_dir)

    early_stop_cb = EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )

    EPOCHS = 15
    BATCH_SIZE = 50

    print(f'Training started... ({EPOCHS} epochs, batch_size={BATCH_SIZE})')
    print(f'Train: {len(train_images)} images, Val: {len(test_images)} images')
    print('-' * 60)

    history = model.fit(
        train_images, train_labels,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1,
        validation_data=(test_images, test_labels),
        callbacks=[tensorboard_cb, early_stop_cb]
    )
    print('Training finished!')


## 8. Visualize Results

In [ ]:
if history.history is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy Graph
    ax1.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
    ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.legend(loc='lower right')
    ax1.grid(True, alpha=0.3)

    # Loss Graph
    ax2.plot(history.history['loss'], label='Train Loss', linewidth=2)
    ax2.plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Loss')
    ax2.set_xlabel('Epoch')
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    os.makedirs(f'{PROJECT_DIR}/output', exist_ok=True)
    plt.savefig(f'{PROJECT_DIR}/output/training_results.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Final results
    print(f"\nFinal Results:")
    print(f"   Train Accuracy: {history.history['accuracy'][-1]:.4f}")
    print(f"   Val Accuracy:   {history.history['val_accuracy'][-1]:.4f}")
    print(f"   Train Loss:     {history.history['loss'][-1]:.4f}")
    print(f"   Val Loss:       {history.history['val_loss'][-1]:.4f}")
else:
    print('Epoch-by-epoch training graphs are unavailable since a pre-trained model was loaded.')
    print('The model is ready! Let\'s evaluate it on the Test dataset: ')
    loss, acc = model.evaluate(test_images, test_labels, verbose=0)
    print(f"-> Accuracy on Test data: {acc*100:.2f}%")
    print(f"-> Test loss: {loss:.4f}")


## 9. Save the Model

In [ ]:
# Save model
model_save_path = f'{PROJECT_DIR}/Code/models'
os.makedirs(model_save_path, exist_ok=True)

model.save(f'{model_save_path}/cats_vs_dogs_cnn.h5')
print(f'Model saved: {model_save_path}/cats_vs_dogs_cnn.h5')

# Save training history
with open(f'{model_save_path}/training_history.pickle', 'wb') as f:
    pickle.dump(history.history, f)
print(f'Training history saved')

## 10. Predictions on Test Images

In [ ]:
def predict_and_show(model, image_paths, num_images=10):
    """Predicts on test images and displays the results"""
    val_map = {0: 'Cat Cat', 1: 'Dog Dog'}
    
    cols = 5
    rows = (num_images + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 3))
    if rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx in range(num_images):
        img_path = os.path.join(CONST.TEST_DIR, image_paths[idx])
        img = cv2.imread(img_path)
        if img is None:
            continue
        img_display = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Preprocessing
        img_resized = cv2.resize(img, (CONST.IMG_SIZE, CONST.IMG_SIZE))
        img_normalized = img_resized.astype('float32') / 255.0
        img_batch = img_normalized.reshape(1, CONST.IMG_SIZE, CONST.IMG_SIZE, 3)
        
        # Prediction
        pred = model.predict(img_batch, verbose=0)
        class_idx = np.argmax(pred[0])
        confidence = np.max(pred[0]) * 100
        
        ax = axes[idx // cols, idx % cols]
        ax.imshow(img_display)
        color = 'green' if confidence > 80 else 'orange'
        ax.set_title(f'{val_map[class_idx]}\n{confidence:.1f}%', 
                     fontsize=12, fontweight='bold', color=color)
        ax.axis('off')
    
    # Hide any empty subplots
    for idx in range(num_images, rows * cols):
        axes[idx // cols, idx % cols].axis('off')
    
    plt.suptitle('Test Predictions', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()


# Select a random sample from the test images
test_images_list = os.listdir(CONST.TEST_DIR)
random.shuffle(test_images_list)
predict_and_show(model, test_images_list, num_images=10)

## 11. Test with Your Own Image

You can upload and test your own image using the cell below.

In [ ]:
# Upload your own image for inference
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
    img = cv2.imread(filename)
    if img is None:
        print(f'{filename} could not be read')
        continue
    
    img_display = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (CONST.IMG_SIZE, CONST.IMG_SIZE))
    img_normalized = img_resized.astype('float32') / 255.0
    img_batch = img_normalized.reshape(1, CONST.IMG_SIZE, CONST.IMG_SIZE, 3)
    
    pred = model.predict(img_batch, verbose=0)
    class_idx = np.argmax(pred[0])
    confidence = np.max(pred[0]) * 100
    
    val_map = {0: 'Cat Cat', 1: 'Dog Dog'}
    
    plt.figure(figsize=(6, 6))
    plt.imshow(img_display)
    plt.title(f'{val_map[class_idx]} — {confidence:.2f}%', fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.show()
    
    print(f'\nResult: {val_map[class_idx]}')
    print(f'Cat probability: {pred[0][0]*100:.2f}%')
    print(f'Dog probability: {pred[0][1]*100:.2f}%')

## 12. Save the Model to Google Drive (Optional)

Save the trained model and history to Google Drive for permanent storage.

In [ ]:
# Save the model to Google Drive for future use
import os
import pickle

save_dir = '/content/drive/MyDrive/ML_Models/cats_vs_dogs'
os.makedirs(save_dir, exist_ok=True)

model.save(f'{save_dir}/cats_vs_dogs_cnn.h5')

# Save training history (for graph) data as well
if history.history is not None:
    with open(f'{save_dir}/training_history.pkl', 'wb') as f:
        pickle.dump(history.history, f)

print(f'Model and results were saved to Google Drive: {save_dir}')
